# Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Loading Dataset

In [2]:
X_train_raw = pd.read_parquet('../data/interim/X_train_raw.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test_raw = pd.read_parquet('../data/interim/X_test_raw.parquet')

In [3]:
X_train_raw.head()

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
id,,,,,,,,,,,,,
0,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,missing,average,sedentary,missing,male


In [4]:
X_test_raw.head()

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
id,,,,,,,,,,,,,
690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,veg,high,poor,active,occasional,male
690089,NaN,83.1,22.42,1773.0,6801.0,24.5,2.40,balanced,high,poor,sedentary,yes,other
690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,balanced,medium,poor,active,no,None
690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,veg,low,good,moderate,yes,other
690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,veg,high,average,active,occasional,other


# Feature Engineering

## Índices de eficiência

In [5]:
X_train_fe = X_train_raw.copy()
X_test_fe = X_test_raw.copy()

In [6]:
eps = 1

X_train_fe["calories_per_min"] = X_train_fe["calorie_expenditure"] / (X_train_fe["exercise_duration"] + eps)
X_test_fe["calories_per_min"] = X_test_fe["calorie_expenditure"] / (X_test_fe["exercise_duration"] + eps)

In [7]:
X_train_fe["steps_per_min"] = X_train_fe["step_count"] / (X_train_fe["exercise_duration"] + eps)
X_test_fe["steps_per_min"] = X_test_fe["step_count"] / (X_test_fe["exercise_duration"] + eps)

In [8]:
X_train_fe["water_per_min"] = X_train_fe["water_intake"] / (X_train_fe["exercise_duration"] + eps)
X_test_fe["water_per_min"] = X_test_fe["water_intake"] / (X_test_fe["exercise_duration"] + eps)

In [9]:
X_train_fe["calories_per_step"] = X_train_fe["calorie_expenditure"] / (X_train_fe["step_count"] + eps)
X_test_fe["calories_per_step"] = X_test_fe["calorie_expenditure"] / (X_test_fe["step_count"] + eps)

## Relação sono × estresse

In [10]:
for data_frame in [X_train_fe, X_test_fe]:
    
    data_frame["cardio_load"] = data_frame["heart_rate"] * data_frame["exercise_duration"]
    data_frame["cardio_efficiency"] = data_frame["calorie_expenditure"] / (data_frame["heart_rate"] + eps)
    data_frame["heart_step"] = data_frame["heart_rate"] / (data_frame["step_count"] + eps)

In [11]:
for data_frame in [X_train_fe, X_test_fe]:
    
    data_frame["bmi_activity"] = data_frame["bmi"] / (data_frame["exercise_duration"] + eps)
    data_frame["bmi_steps"] = data_frame["bmi"] / (data_frame["step_count"] + eps)
    data_frame["bmi_calorie"] = data_frame["bmi"] / (data_frame["calorie_expenditure"] + eps)

## Interações entre sono e estresse

In [12]:
# A. Interação entre 'sleep_quality' e 'stress_level'
# Multiplicação ou soma das codificações numéricas pode capturar se ter um sono ruim E alto estresse é pior do que um único fator.
# Exemplo (assumindo que stress_level e sleep_quality foram convertidos para numéricos ordinais):
X_train_fe['sleep_stress_interaction'] = X_train_fe['sleep_quality'].astype(str) + '_' + X_train_fe['stress_level'].astype(str)
X_test_fe['sleep_stress_interaction'] = X_test_fe['sleep_quality'].astype(str) + '_' + X_test_fe['stress_level'].astype(str)

## Ratios de Features

In [13]:
# Criar novas features combinando outras, como a relação entre 'water_intake' e 'calorie_expenditure' ou 'age' e 'bmi'.
# Exemplo:
X_train_fe['water_to_calorie_ratio'] = X_train_fe['water_intake'] / (X_train_fe['calorie_expenditure'] + eps)
X_test_fe['water_to_calorie_ratio'] = X_test_fe['water_intake'] / (X_test_fe['calorie_expenditure'] + eps)

## NA for Categorical Feat.

In [14]:
cat_features_raw = [
    'diet_type',
    'stress_level',
    'sleep_quality',
    'physical_activity_level',
    'smoking_alcohol',
    'gender'
]

In [15]:
for cat in cat_features_raw:

    for data_frame in [X_train_raw, X_test_raw]:
        
        data_frame[cat] = data_frame[cat].fillna("missing").astype("category")

In [16]:
cat_features_fe = [
    'diet_type',
    'stress_level',
    'sleep_quality',
    'physical_activity_level',
    'smoking_alcohol',
    'gender',
    'sleep_stress_interaction'
]

In [17]:
for cat in cat_features_fe:

    for data_frame in [X_train_fe, X_test_fe]:
        
        data_frame[cat] = data_frame[cat].fillna("missing").astype("category")

# Saving Databases

In [18]:
X_train_fe.dtypes

sleep_duration               float64
heart_rate                   float64
bmi                          float64
calorie_expenditure          float64
step_count                   float64
exercise_duration            float64
water_intake                 float64
diet_type                   category
stress_level                category
sleep_quality               category
physical_activity_level     category
smoking_alcohol             category
gender                      category
calories_per_min             float64
steps_per_min                float64
water_per_min                float64
calories_per_step            float64
cardio_load                  float64
cardio_efficiency            float64
heart_step                   float64
bmi_activity                 float64
bmi_steps                    float64
bmi_calorie                  float64
sleep_stress_interaction    category
water_to_calorie_ratio       float64
dtype: object

In [19]:
X_train_fe.isna().mean()

sleep_duration              0.110129
heart_rate                  0.011351
bmi                         0.020139
calorie_expenditure         0.076589
step_count                  0.020166
exercise_duration           0.010000
water_intake                0.063002
diet_type                   0.000000
stress_level                0.000000
sleep_quality               0.000000
physical_activity_level     0.000000
smoking_alcohol             0.000000
gender                      0.000000
calories_per_min            0.085862
steps_per_min               0.029976
water_per_min               0.072360
calories_per_step           0.095202
cardio_load                 0.021251
cardio_efficiency           0.087056
heart_step                  0.031296
bmi_activity                0.029951
bmi_steps                   0.039920
bmi_calorie                 0.095204
sleep_stress_interaction    0.000000
water_to_calorie_ratio      0.134670
dtype: float64

In [20]:
X_train_raw.dtypes

sleep_duration              float64
heart_rate                  float64
bmi                         float64
calorie_expenditure         float64
step_count                  float64
exercise_duration           float64
water_intake                float64
diet_type                  category
stress_level               category
sleep_quality              category
physical_activity_level    category
smoking_alcohol            category
gender                     category
dtype: object

In [21]:
X_train_raw.isna().mean()

sleep_duration             0.110129
heart_rate                 0.011351
bmi                        0.020139
calorie_expenditure        0.076589
step_count                 0.020166
exercise_duration          0.010000
water_intake               0.063002
diet_type                  0.000000
stress_level               0.000000
sleep_quality              0.000000
physical_activity_level    0.000000
smoking_alcohol            0.000000
gender                     0.000000
dtype: float64

In [22]:
X_train_raw.to_parquet('../data/processed/X_train_raw.parquet', engine='fastparquet')
X_test_raw.to_parquet('../data/processed/X_test_raw.parquet', engine='fastparquet')

In [23]:
X_train_fe.to_parquet('../data/processed/X_train_fe.parquet', engine='fastparquet')
X_test_fe.to_parquet('../data/processed/X_test_fe.parquet', engine='fastparquet')

In [24]:
X_train_fe.shape

(690088, 25)

In [25]:
X_test_fe.shape

(295753, 25)